# GPT-2 Cross-Linguistic Consistency Check

Runs the same corrected token-reveal pipeline with **GPT-2 (124M)** on all multilingual Wikipedia
datasets + Buckeye + French, to test whether the exponent is consistent across languages
within a single probe model.

**Known GPT-2 RAID result**: human α = -0.77

**Question**: Does GPT-2 give ~-0.77 for Chinese, Japanese, Korean, Turkish, Arabic, Finnish,
Buckeye spoken, and French spoken too? If yes, the within-probe consistency is the real finding.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_BASE = Path('/content/drive/MyDrive/LRTIA/Data')
    BASE_DIR = Path('/content/drive/MyDrive/LRTIA/Results/GPT2_crosslingual')
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_BASE = Path('../data')
    BASE_DIR = Path('../results/GPT2_crosslingual')
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'gpt2'
MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42

# All datasets to process
DATASETS = {}

# Multilingual Wikipedia
lang_names = {'zh': 'Chinese', 'ja': 'Japanese', 'ko': 'Korean',
              'tr': 'Turkish', 'ar': 'Arabic', 'fi': 'Finnish'}
for lang, name in lang_names.items():
    p = DATA_BASE / 'wiki_multilingual' / f'{lang}_articles.jsonl'
    if p.exists():
        DATASETS[f'wiki_{lang}'] = {'path': p, 'label': f'{name} Wiki', 'family': lang}

# Buckeye
bk = DATA_BASE / 'buckeye_processed' / 'speaker_concatenated.jsonl'
if bk.exists():
    DATASETS['buckeye'] = {'path': bk, 'label': 'Buckeye spoken', 'family': 'en_spoken'}

# French
fr = DATA_BASE / 'french_oral_processed' / 'per_story.jsonl'
if fr.exists():
    DATASETS['french'] = {'path': fr, 'label': 'French spoken', 'family': 'fr_spoken'}

print(f'Datasets found: {len(DATASETS)}')
for k, v in DATASETS.items():
    print(f'  {v["label"]}')

In [ ]:
# === Load GPT-2 ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print(f'Loaded {MODEL_NAME} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params)')

In [ ]:
# === Core functions ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')

def compute_token_reveal_curve(full_ids, target_start, target_end,
                                shuffled=False, rng_shuf=None):
    target_ids = full_ids[target_start:target_end]
    context_pool = list(full_ids[:target_start])
    if shuffled and rng_shuf is not None:
        context_pool = list(context_pool)
        rng_shuf.shuffle(context_pool)
    max_ctx = min(MAX_CONTEXT, len(context_pool))
    if max_ctx < 10:
        return None
    ppls, ctx_lengths = [], []
    for ctx_len in range(1, max_ctx + 1):
        ctx_tokens = context_pool[-ctx_len:]
        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)
    if len(ppls) < 10:
        return None
    return {'ctx_lengths': ctx_lengths, 'ppls': ppls}

def process_document(doc, rng_shuf):
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    intact_curves, shuffled_curves = [], []
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5:
            continue
        result = compute_token_reveal_curve(full_ids, target_start, target_end)
        if result is not None:
            result['doc_id'] = doc.get('doc_id', '')
            result['target_frac'] = frac
            intact_curves.append(result)
        result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                              shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            result_s['doc_id'] = doc.get('doc_id', '')
            result_s['target_frac'] = frac
            shuffled_curves.append(result_s)
    return intact_curves, shuffled_curves

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

print('Functions defined')

In [ ]:
# === Process all datasets ===
all_results = {}  # key -> {intact, shuffled, slope, r, p}

for key, info in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'{info["label"]} ({key})')
    print(f'{"="*60}')

    # Check cache
    intact_path = BASE_DIR / f'{key}_intact.json'
    shuffled_path = BASE_DIR / f'{key}_shuffled.json'

    if intact_path.exists() and shuffled_path.exists():
        with open(intact_path) as f:
            intact = json.load(f)
        with open(shuffled_path) as f:
            shuffled = json.load(f)
        print(f'  Loaded {len(intact)} intact + {len(shuffled)} shuffled from cache')
    else:
        corpus = []
        with open(info['path']) as f:
            for line in f:
                corpus.append(json.loads(line))
        print(f'  Loaded {len(corpus)} documents')

        intact, shuffled = [], []
        rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
        for doc in tqdm(corpus, desc=info['label']):
            i, s = process_document(doc, rng_shuf)
            intact.extend(i)
            shuffled.extend(s)

        with open(intact_path, 'w') as f:
            json.dump(intact, f)
        with open(shuffled_path, 'w') as f:
            json.dump(shuffled, f)
        print(f'  Computed {len(intact)} intact + {len(shuffled)} shuffled')

    # Fit
    if len(intact) >= 5 and len(shuffled) >= 5:
        ip = compute_raw_ppl_curve(intact)
        sp = compute_raw_ppl_curve(shuffled)
        corr = -np.diff(ip) - (-np.diff(sp))
        fit = fit_power_law(corr)
        if fit:
            all_results[key] = {
                'label': info['label'], 'slope': fit[0], 'r': fit[1], 'p': fit[2],
                'n': len(intact), 'corrected_marg': corr,
                'bc': fit[3], 'bm': fit[4], 'intercept': fit[5],
            }
            print(f'  >> α = {fit[0]:.3f} (r={fit[1]:.3f}, p={fit[2]:.4f})')
        else:
            print(f'  >> Fit failed')
    else:
        print(f'  >> Not enough curves')

print(f'\n\nAll done! {len(all_results)} datasets with successful fits')

In [ ]:
# === Add RAID reference (from multi-model notebook) ===
# GPT-2 on RAID human was α = -0.765
RAID_REF = {'label': 'RAID English (ref)', 'slope': -0.765, 'r': -0.946}

# === Summary table ===
print(f'{"Dataset":<25} {"α":>8} {"r":>8} {"N":>6}')
print('-' * 50)
print(f'{RAID_REF["label"]:<25} {RAID_REF["slope"]:>8.3f} {RAID_REF["r"]:>8.3f} {"—":>6}')
for key, res in all_results.items():
    print(f'{res["label"]:<25} {res["slope"]:>8.3f} {res["r"]:>8.3f} {res["n"]:>6}')

# Stats
all_exps = [RAID_REF['slope']] + [r['slope'] for r in all_results.values()]
strong_exps = [RAID_REF['slope']] + [r['slope'] for r in all_results.values() if r['r'] < -0.80]
print(f'\nAll datasets: mean={np.mean(all_exps):.3f}, SD={np.std(all_exps):.3f}, '
      f'range=[{min(all_exps):.3f}, {max(all_exps):.3f}]')
if len(strong_exps) > 1:
    print(f'Strong fits (r<-0.80): mean={np.mean(strong_exps):.3f}, SD={np.std(strong_exps):.3f}, '
          f'range=[{min(strong_exps):.3f}, {max(strong_exps):.3f}]')

In [ ]:
# === Figure: GPT-2 exponents across all languages ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Bar chart of exponents
ax = axes[0]
labels = [RAID_REF['label']] + [r['label'] for r in all_results.values()]
exponents = [RAID_REF['slope']] + [r['slope'] for r in all_results.values()]
r_vals = [RAID_REF['r']] + [r['r'] for r in all_results.values()]
colors = ['blue'] + ['#2ca02c'] * len(all_results)

bars = ax.bar(range(len(labels)), exponents, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9, rotation=45, ha='right')
ax.set_ylabel('Power Law Exponent (α)', fontsize=12)
ax.set_title('GPT-2 (124M): Exponent Across Languages', fontweight='bold')
ax.axhline(np.mean(exponents), color='black', linestyle='-', linewidth=2, alpha=0.5,
           label=f'Mean: {np.mean(exponents):.2f}')
ax.axhline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')
for i, (exp, r) in enumerate(zip(exponents, r_vals)):
    ax.text(i, exp - 0.04, f'{exp:.2f}\n(r={r:.2f})', ha='center', fontsize=7, fontweight='bold')

# Panel B: Overlaid corrected marginals
ax = axes[1]
for key, res in all_results.items():
    ax.plot(common_x[1:], uniform_filter1d(res['corrected_marg'], 5),
            '-', linewidth=1.5, alpha=0.7, label=f'{res["label"]}: α={res["slope"]:.2f}')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal', fontsize=12)
ax.set_title('GPT-2: Coherence Signal Across Languages', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.2)

plt.suptitle('Within-Probe Consistency: GPT-2 (124M) Across 8 Languages + Spoken',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_gpt2_crosslingual.png', dpi=150, bbox_inches='tight')
plt.show()

## Interpretation

If the GPT-2 exponents cluster tightly across languages (similar SD to the Mistral results),
then the within-probe consistency is real regardless of which probe you use.

**Mistral within-probe**: mean=-0.71, SD=0.08 (7 strong-fit datasets)

**GPT-2 within-probe**: see above

If both probes show tight clustering (even at different absolute values), the paper's claim is:
coherence decay rate is a stable property of human language that different probes measure
with different sensitivity but consistent relative values.